In [7]:
import torch
import os
import pandas as pd
import librosa
import matplotlib.pyplot as plt

from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch import nn
from sklearn.model_selection import KFold
from torch.utils.data import SubsetRandomSampler
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from torch.nn.functional import log_softmax

In [ ]:
class PhonationTypeDataset(Dataset):
    """Dataset including phonation type samples of uniform length.
    
    Attributes:
        audio_dir: Directory containing the samples as wave files.
        sample_rate: Audio sample rate used by the transformer (16,000 Hz is usually best).
        transform: Transform conducted to the audio (not implemented).
        target_transform: Transform conducted to the labels (not implemented).
    """
    def __init__(self, audio_dir, sample_rate=16000, transform=None, target_transform=None):
        self.audio_dir = audio_dir
        self.sample_rate = sample_rate
        self.transform = transform
        self.target_transform = target_transform

        file_names = []
        labels = []

        for filename in os.listdir(audio_dir):

            if "normal" in filename:
                file_names.append(filename)
                labels.append(0)
            elif "breathy" in filename:
                file_names.append(filename)
                labels.append(1)
            elif "pressed" in filename:
                file_names.append(filename)
                labels.append(2)

        self.audio_labels = pd.DataFrame({"file_name": file_names, "label": labels})

    def __len__(self):
        return len(self.audio_labels)

    def __getitem__(self, idx):

        file_path = os.path.join(self.audio_dir, self.audio_labels.iloc[idx, 0])
        audio_array, sr = librosa.load(file_path, sr=self.sample_rate)
        label = self.audio_labels.iloc[idx, 1]
        """
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        """
        return audio_array, label

In [4]:
dataset = PhonationTypeDataset(audio_dir="trimmed_data", sample_rate=16000)

print("Dataset length", len(dataset))
print("Instance size", dataset[0][0].shape)

Dataset length 1140
Instance size (15000,)


In [5]:
print("Normal:", len(dataset.audio_labels.loc[dataset.audio_labels["label"] == 0]))
print("Breathy:", len(dataset.audio_labels.loc[dataset.audio_labels["label"] == 1]))
print("Pressed:", len(dataset.audio_labels.loc[dataset.audio_labels["label"] == 2]))

Normal: 380
Breathy: 380
Pressed: 380


In [ ]:
class SLFN(nn.Module):
    """A single-hidden-layer feed forward neural network used as a classifier head.
    """
    def __init__(self, input_size, hidden_size):
        super(SLFN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 4)
        )
    
    def forward(self, x):

        y = self.model(x)

        return y

In [ ]:
"""Training loop.
"""

folds = 10 # 10 fold cross validation
batch_size = 32 # dataset size 1140
epochs = 50 # taken over by early stopping
layer_number = 2 # used transformer layer used to extract feature vectors

kf = KFold(n_splits=folds, shuffle=True)
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h") # alkiskoudounas/voc2vec-ls-pt facebook/wav2vec2-base-960h
model = AutoModelForAudioClassification.from_pretrained("facebook/wav2vec2-base-960h", output_hidden_states=True)

fold_accuracies = list()

for i, (train_idx, test_idx) in enumerate(kf.split(dataset)):
    print("fold:", i+1, "/", folds)

    train_loader = DataLoader(dataset=dataset, batch_size=batch_size, sampler=SubsetRandomSampler(train_idx))
    test_loader = DataLoader(dataset=dataset, batch_size=batch_size, sampler=SubsetRandomSampler(test_idx))

    # train(model) with train_loader ->

    classifier = SLFN(input_size=768, hidden_size=1024)
    optimizer = torch.optim.Adam(classifier.parameters(), lr=0.001)
    loss_function = nn.NLLLoss()

    # variables for early stopping
    high_accuracy = 0
    h = 0

    for epoch_idx in range(epochs):
        print("epoch:", epoch_idx+1)

        # train each epoch
        for audio_array_batch, labels in train_loader:

            features = feature_extractor(audio_array_batch, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt")
            inputs = torch.flatten(features.input_values, end_dim=1)
            model_outputs = model(inputs)
            layer_outputs = model_outputs.hidden_states[layer_number]
            average_features = torch.mean(layer_outputs, dim=1)

            optimizer.zero_grad()
            output = classifier(average_features)
            loss = loss_function(log_softmax(output, dim=1), labels)
            loss.backward()
            optimizer.step()
        
        # evaluate for each epoch
        correct = 0
        total = 0

        for audio_array_batch, labels in test_loader:

            features = feature_extractor(audio_array_batch, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt")
            inputs = torch.flatten(features.input_values, end_dim=1)
            model_outputs = model(inputs)
            layer_outputs = model_outputs.hidden_states[layer_number]
            average_features = torch.mean(layer_outputs, dim=1)

            output = classifier(average_features)
            logits = log_softmax(output, dim=1)
            predictions = torch.argmax(logits, dim=1)

            correct += torch.sum(predictions == labels)
            total += len(labels)
        
        epoch_accuracy = correct / total
        
        # early stopping
        if epoch_accuracy > high_accuracy:
            h = 0
            high_accuracy = epoch_accuracy
        elif h == 4:
            break
        else:
            h += 1


    # evaluate(model) with test_loader

    correct = 0
    total = 0

    for audio_array_batch, labels in test_loader:

        features = feature_extractor(audio_array_batch, sampling_rate=feature_extractor.sampling_rate, return_tensors="pt")
        inputs = torch.flatten(features.input_values, end_dim=1)
        model_outputs = model(inputs)
        layer_outputs = model_outputs.hidden_states[layer_number]
        average_features = torch.mean(layer_outputs, dim=1)

        output = classifier(average_features)
        logits = log_softmax(output, dim=1)
        predictions = torch.argmax(logits, dim=1)

        correct += torch.sum(predictions == labels)
        total += len(labels)
    
    fold_accuracy = correct / total
    fold_accuracies.append(fold_accuracy)
    print("fold", i+1, "accuracy:", fold_accuracy.item())


Loading weights: 100%|██████████| 210/210 [00:00<00:00, 4747.42it/s]
[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.weight             | UNEXPECTED | 
wav2vec2.masked_spec_embed | MISSING    | 
projector.bias             | MISSING    | 
classifier.bias            | MISSING    | 
projector.weight           | MISSING    | 
classifier.weight          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


fold: 1 / 10
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
fold 1 accuracy: 0.5877193212509155
fold: 2 / 10
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
fold 2 accuracy: 0.5526315569877625
fold: 3 / 10
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
epoch: 21
epoch: 22
epoch: 23
epoch: 24
epoch: 25
epoch: 26
epoch: 27
fold 3 accuracy: 0.6842105388641357
fold: 4 / 10
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
epoch: 8
epoch: 9
epoch: 10
epoch: 11
epoch: 12
epoch: 13
epoch: 14
epoch: 15
epoch: 16
epoch: 17
epoch: 18
epoch: 19
epoch: 20
fold 4 accuracy: 0.6842105388641357
fold: 5 / 10
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
epoch: 7
fold 5 accuracy: 0.5087719559669495
fold: 6 / 10
epoch: 1
epoch: 2
epoch: 3
epoch: 4
epoch: 5
epoch: 6
e

In [ ]:
for i, accuracy in enumerate(fold_accuracies):
    print("Fold", str(i+1), "accuracy:", str(accuracy.item()))

print("Average:", str(torch.mean(torch.tensor(fold_accuracies)).item()))

Fold 1 accuracy: 0.5877193212509155
Fold 2 accuracy: 0.5526315569877625
Fold 3 accuracy: 0.6842105388641357
Fold 4 accuracy: 0.6842105388641357
Fold 5 accuracy: 0.5087719559669495
Fold 6 accuracy: 0.5350877046585083
Fold 7 accuracy: 0.5526315569877625
Fold 8 accuracy: 0.5964912176132202
Fold 9 accuracy: 0.6140350699424744
Fold 10 accuracy: 0.5789473652839661
Average: 0.5894736647605896


Transformer layer 0
Fold 1 accuracy: 0.429824560880661
Fold 2 accuracy: 0.6052631735801697
Fold 3 accuracy: 0.6052631735801697
Fold 4 accuracy: 0.6140350699424744
Fold 5 accuracy: 0.6140350699424744
Fold 6 accuracy: 0.5
Fold 7 accuracy: 0.6754385828971863
Fold 8 accuracy: 0.6842105388641357
Fold 9 accuracy: 0.5526315569877625
Fold 10 accuracy: 0.5789473652839661
Average: 0.5859648585319519

Layer 1
Fold 1 accuracy: 0.5877193212509155
Fold 2 accuracy: 0.5789473652839661
Fold 3 accuracy: 0.6578947305679321
Fold 4 accuracy: 0.6315789222717285
Fold 5 accuracy: 0.5789473652839661
Fold 6 accuracy: 0.6315789222717285
Fold 7 accuracy: 0.5350877046585083
Fold 8 accuracy: 0.6140350699424744
Fold 9 accuracy: 0.5789473652839661
Fold 10 accuracy: 0.5701754093170166
Average: 0.5964912176132202

Layer 2
Fold 1 accuracy: 0.5877193212509155
Fold 2 accuracy: 0.5526315569877625
Fold 3 accuracy: 0.6842105388641357
Fold 4 accuracy: 0.6842105388641357
Fold 5 accuracy: 0.5087719559669495
Fold 6 accuracy: 0.5350877046585083
Fold 7 accuracy: 0.5526315569877625
Fold 8 accuracy: 0.5964912176132202
Fold 9 accuracy: 0.6140350699424744
Fold 10 accuracy: 0.5789473652839661
Average: 0.5894736647605896